# ✅ First-Order Approximation of Reward Verification
This notebook verifies the approximation:

$$
\hat{r}_\theta(x, y) \approx \hat{r}_{\theta^*}(x, y) + \beta \nabla_{\theta^*} \log \pi_{\theta^*}(y \mid x)^\top (\theta - \theta^*)
$$

We use a HuggingFace causal language model and test whether the reward (log-prob of label) under perturbation can be well-approximated by a first-order expansion.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import copy
import matplotlib.pyplot as plt

## 🛠️ Core Functions

In [ ]:
def compute_logprob(model, input_ids, labels):
    outputs = model(input_ids=input_ids, labels=labels)
    log_probs = -outputs.loss * labels.shape[1]
    return log_probs

def compute_gradient_logprob(model, input_ids, labels):
    model.zero_grad()
    outputs = model(input_ids=input_ids, labels=labels)
    loss = outputs.loss
    loss.backward()
    grads = []
    for param in model.parameters():
        if param.grad is not None:
            grads.append(param.grad.view(-1))
    return torch.cat(grads)

def perturb_model(model, std):
    deltas = []
    for param in model.parameters():
        delta = torch.randn_like(param) * std
        param.data += delta
        deltas.append(delta.view(-1))
    return torch.cat(deltas)

## 🔬 Single-sample First-Order Approximation Validation

In [ ]:
def validate_first_order_reward(model_name="gpt2", prompt="The meaning of life is", beta=1.0, perturb_std=1e-3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
    model.eval()
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    input_ids = inputs["input_ids"]
    labels = input_ids.clone()

    with torch.no_grad():
        r_star = compute_logprob(model, input_ids, labels)

    grad_logp = compute_gradient_logprob(model, input_ids, labels)
    delta = perturb_model(model, perturb_std)

    with torch.no_grad():
        r_theta = compute_logprob(model, input_ids, labels)

    r_approx = r_star + beta * grad_logp.dot(delta)

    print(f"True Δr:        {(r_theta - r_star).item():.6f}")
    print(f"First-order Δr: {(r_approx - r_star).item():.6f}")
    print(f"Absolute error: {abs((r_theta - r_star - (r_approx - r_star)).item()):.6f}")

## 📈 Run Multiple Samples and Plot Approximation Error

In [ ]:
def experiment_on_samples(model_name="gpt2", prompts=None, perturb_std=1e-3):
    if prompts is None:
        prompts = [
            "The capital of France is",
            "Once upon a time",
            "The answer to life is",
            "The president of USA is",
            "In a galaxy far away,"
        ]
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
    model.eval()
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token

    errors = []

    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        input_ids = inputs["input_ids"]
        labels = input_ids.clone()

        with torch.no_grad():
            r_star = compute_logprob(model, input_ids, labels)
        grad_logp = compute_gradient_logprob(model, input_ids, labels)
        delta = perturb_model(model, perturb_std)

        with torch.no_grad():
            r_theta = compute_logprob(model, input_ids, labels)
        r_approx = r_star + grad_logp.dot(delta)

        error = abs((r_theta - r_star - (r_approx - r_star)).item())
        errors.append(error)

    plt.figure(figsize=(8, 5))
    plt.plot(errors, marker='o')
    plt.title("First-order Approximation Error over Samples")
    plt.xlabel("Sample Index")
    plt.ylabel("Absolute Approximation Error")
    plt.grid(True)
    plt.show()

## ▶️ Run Single or Multiple Verifications Below

In [ ]:
# Single test
validate_first_order_reward(model_name="gpt2", prompt="The capital of France is")

# Multi-sample test
experiment_on_samples(model_name="gpt2", perturb_std=1e-3)